[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nekrut/bda/blob/colab/lectures/lecture7.ipynb)

# Lecture 7: Data Manipulation with Pandas

> This tutorial draws on material from:
> - [Justin Bois](http://justinbois.github.io/bootcamp/2020/index.html)
> - [BIOS821 course at Duke](https://people.duke.edu/~ccc14/bios-821-2017/index.html)
> - [Pandas documentation](https://pandas.pydata.org/docs/user_guide/index.html/)

Pandas (from "Panel Data") is a library for manipulating tabular data in Python. Its central object—the **DataFrame**—lets you filter, transform, aggregate, and reshape tables with concise, readable code. We will use a small genome statistics dataset throughout this lecture.

In [ ]:
import pandas as pd

## Creating a DataFrame

The simplest way to create a DataFrame is from a Python dictionary. Each key becomes a column name, and the corresponding list becomes the column values.

In [ ]:
small_df = pd.DataFrame({
    'organism': ['E. coli', 'S. cerevisiae', 'H. sapiens'],
    'genome_size_mb': [4.6, 12.1, 3100.0],
    'num_genes': [4300, 6000, 20000]
})
small_df

In [ ]:
print('Shape:', small_df.shape)
print()
print(small_df.dtypes)

### Exercise

Create a DataFrame called `viruses` with 3 columns: `name`, `genome_size_kb`, and `host`. Include at least 3 rows of made-up or real virus data. Print its shape.

In [ ]:
# Your code here

## Reading CSV files

In practice, data lives in files. `pd.read_csv()` reads a CSV into a DataFrame. Here we load a genome statistics dataset:

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/nekrut/bda/main/lectures/data/genome_stats.csv')
df.head()

In [ ]:
print('Shape:', df.shape)
print()
print(df.dtypes)

### Exercise

Use `.tail()` to show the last 3 rows of `df`. Then use `.describe()` to get summary statistics. Which organism has the largest genome?

In [ ]:
# Your code here

## Indexing

There are three main ways to select data from a DataFrame:

- **`df['col']`** — select a single column (returns a Series)
- **`df.loc[row_label, col_label]`** — label-based indexing (uses actual row/column names)
- **`df.iloc[row_pos, col_pos]`** — integer position-based indexing

In [ ]:
# Select a single column
df['organism'].head()

In [ ]:
# Label-based: row 3, column 'gc_percent'
df.loc[3, 'gc_percent']

In [ ]:
# Integer position-based: rows 2 through 4
df.iloc[2:5]

### Exercise

Using `df.loc`, extract the `organism` and `habitat` columns for rows 5 through 9. Then use `df.iloc` to get the value in the 4th row, 3rd column.

In [ ]:
# Your code here

## Filtering

Boolean indexing lets you select rows that match a condition. The expression inside the brackets produces a True/False Series, and only rows where the value is True are kept.

> **Note:** `df[condition]` is shorthand for `df.loc[condition]`—both are acceptable. We use the shorter form here for readability.

In [ ]:
# All bacteria
df[df['kingdom'] == 'Bacteria']

In [ ]:
# Bacteria with GC content above 50%
df[(df['kingdom'] == 'Bacteria') & (df['gc_percent'] > 50)]

### Exercise

Filter `df` to show only organisms that live in `"aquatic"` habitats. Then find all Eukarya with more than 20,000 genes.

In [ ]:
# Your code here

## Adding columns

You can create new columns from existing ones. Pandas applies operations **element-wise** (vectorization)—no `for` loop needed.

> **Note:** This permanently adds the column to `df`. All subsequent cells will see 7 columns instead of 6.

In [ ]:
df['genes_per_mb'] = df['num_genes'] / df['genome_size_mb']
df[['organism', 'genome_size_mb', 'num_genes', 'genes_per_mb']].head()

### Exercise

Add a column `gc_category` that is `"high"` when `gc_percent` > 50 and `"low"` otherwise. Hint: use `np.where()` — you'll need `import numpy as np`.

In [ ]:
# Your code here

## Writing CSV

Use `df.to_csv()` to save a DataFrame back to disk. Setting `index=False` prevents Pandas from writing the row numbers as an extra unnamed column—without it, the CSV gets cluttered with a meaningless index.

In [ ]:
df.to_csv('genome_stats_with_density.csv', index=False)
# Verify round-trip
pd.read_csv('genome_stats_with_density.csv').head(3)

## Exploring categories

Before discussing data organization principles, let's explore what categories exist in our dataset.

In [ ]:
df['kingdom'].unique()

In [ ]:
df['kingdom'].value_counts()

In [ ]:
df['habitat'].value_counts()

### Exercise

How many unique habitats are there? Which habitat has the fewest organisms? Use `.unique()` and `.value_counts()`.

In [ ]:
# Your code here

---

# Tidy data

[Hadley Wickham](https://en.wikipedia.org/wiki/Hadley_Wickham) defined [tidy data](http://dx.doi.org/10.18637/jss.v059.i10) with three rules:

1. Each **variable** is a column.
2. Each **observation** is a row.
3. Each **type of observation** has its own separate data frame.

Tidy data is far easier to filter, aggregate, and plot.

Our `genome_stats.csv` is already tidy: each row is one organism (an observation), each column is a single variable (kingdom, genome size, etc.), and all rows describe the same type of thing—genome statistics.

## Wide vs long format

Data often arrives in **wide** format—one column per experimental condition. This is convenient for spreadsheets but violates tidy principles because the column headers contain data (condition names).

Below is a small gene expression dataset. The values represent expression levels measured in FPKM (Fragments Per Kilobase of transcript per Million mapped reads)—a common unit for RNA-seq experiments.

In [ ]:
expr = pd.read_csv('https://raw.githubusercontent.com/nekrut/bda/main/lectures/data/gene_expression_wide.csv')
expr

The condition columns (`control`, `heat_shock`, `starvation`) are really *values* of a variable we might call `condition`. We use `pd.melt()` to reshape this into tidy (long) format.

![Pandas melt operation diagram](https://pandas.pydata.org/docs/_images/07_melt.svg)

Key parameters of `pd.melt()`:
- **`id_vars`** — columns to keep as identifiers (not melted)
- **`var_name`** — name for the new column created from the old column headers
- **`value_name`** — name for the new column holding the cell values

In [ ]:
expr_long = pd.melt(
    expr,
    id_vars=['gene'],
    var_name='condition',
    value_name='expression'
)
expr_long

To go back from long to wide, use `.pivot()`:

![Pandas pivot operation diagram](https://pandas.pydata.org/docs/_images/07_pivot.svg)

In [ ]:
expr_long.pivot(index='gene', columns='condition', values='expression')

### Exercise

The DataFrame below is in wide format. Melt it into tidy (long) format, then pivot it back to wide.

```python
temps = pd.DataFrame({
    'city': ['NYC', 'LA', 'Chicago'],
    'jan': [-1, 14, -5],
    'jul': [28, 24, 27]
})
```

In [ ]:
# Your code here

---

# Sorting

Use `sort_values()` to order rows by one or more columns. By default sorting is ascending; pass `ascending=False` to reverse.

In [ ]:
df.sort_values('genome_size_mb')

In [ ]:
# Multi-column sort: kingdom ascending, genome size descending
df.sort_values(
    by=['kingdom', 'genome_size_mb'],
    ascending=[True, False]
)

### Exercise

Sort `df` by `num_genes` in descending order. Which organism has the most genes? Then sort by `habitat` ascending and `gc_percent` descending.

In [ ]:
# Your code here

---

# Split-apply-combine

Many analyses follow a three-step pattern described by [Hadley Wickham](http://dx.doi.org/10.18637/jss.v040.i01):

1. **Split** the data into groups (e.g., by kingdom)
2. **Apply** a function to each group (e.g., compute the mean)
3. **Combine** the results into a new table

In Pandas this is done with `groupby()`.

## Aggregation

In [ ]:
df.groupby('kingdom')['genome_size_mb'].mean()

In [ ]:
df.groupby('kingdom')['genome_size_mb'].describe()

In [ ]:
# Group by two columns
df.groupby(['kingdom', 'habitat'])['num_genes'].mean()

### Exercise

Use `groupby` to find the **maximum** `gc_percent` for each kingdom. Then compute the mean `num_genes` grouped by `habitat`.

In [ ]:
# Your code here

---

# Working with multiple tables

Real analyses often require combining information from different sources. `pd.merge()` joins two DataFrames on a shared key—similar to SQL joins.

![Pandas merge left join diagram](https://pandas.pydata.org/docs/_images/08_merge_left.svg)

Let's create two small DataFrames to demonstrate different join types.

In [ ]:
taxonomy = pd.DataFrame({
    'organism': ['Escherichia coli', 'Saccharomyces cerevisiae',
                 'Homo sapiens', 'Halobacterium salinarum'],
    'phylum': ['Pseudomonadota', 'Ascomycota',
               'Chordata', 'Euryarchaeota']
})
taxonomy

In [ ]:
isolation = pd.DataFrame({
    'organism': ['Escherichia coli', 'Bacillus subtilis',
                 'Homo sapiens', 'Drosophila melanogaster'],
    'first_sequenced': [1997, 1997, 2001, 2000]
})
isolation

### Inner join

Keeps only rows whose key appears in **both** tables.

In [ ]:
pd.merge(taxonomy, isolation, on='organism')

### Left join

Keeps all rows from the **left** table; fills missing matches with NaN.

In [ ]:
pd.merge(taxonomy, isolation, on='organism', how='left')

> **What is NaN?** NaN stands for "Not a Number" and represents missing data. When a left join finds no matching row in the right table, Pandas fills those cells with NaN.

### Right join

Keeps all rows from the **right** table.

In [ ]:
pd.merge(taxonomy, isolation, on='organism', how='right')

### Outer join

Keeps all rows from **both** tables.

In [ ]:
pd.merge(taxonomy, isolation, on='organism', how='outer')

### Exercise

Create two DataFrames and merge them:

```python
habitats = pd.DataFrame({
    'habitat': ['gut', 'soil', 'aquatic'],
    'temperature_c': [37, 20, 15]
})
```

Merge `habitats` with `df` using a left join on `'habitat'`. How many rows have NaN for `temperature_c`? Why?

In [ ]:
# Your code here

---

# Visualization with Altair

Let's make a quick bar chart of mean genome size by kingdom using `groupby` + Altair.

Altair uses single-letter encoding type suffixes:
- **`:N`** — nominal (categorical, unordered) data like kingdom names
- **`:Q`** — quantitative (numeric, continuous) data like genome size
- **`:O`** — ordinal (ordered categories) and **`:T`** — temporal (dates/times)

In [ ]:
import altair as alt

kingdom_means = df.groupby('kingdom')['genome_size_mb'].mean().reset_index()

alt.Chart(kingdom_means).mark_bar().encode(
    x=alt.X('kingdom:N', title='Kingdom'),
    y=alt.Y('genome_size_mb:Q', title='Mean genome size (Mb)'),
    color='kingdom:N'
).properties(
    width=300,
    title='Mean Genome Size by Kingdom'
)

### Exercise

Create a scatter plot of `genome_size_mb` (x) vs `num_genes` (y), colored by `kingdom`. Use `mark_point()` instead of `mark_bar()`.

In [ ]:
# Your code here

# Summary

- **DataFrame basics**: create from dict, read CSV, inspect with `.head()`, `.shape`, `.dtypes`
- **Indexing**: `[]`, `.loc` (label), `.iloc` (position)
- **Filtering**: boolean indexing with `&` for compound conditions
- **New columns**: vectorized arithmetic—no loops needed
- **Tidy data**: each variable a column, each observation a row
- **Reshaping**: `melt()` (wide → long), `pivot()` (long → wide)
- **Sorting**: `sort_values()` with multi-column and mixed ascending/descending
- **Split-apply-combine**: `groupby()` → `.mean()`, `.describe()`, `.agg()`
- **Joins**: `pd.merge()` with inner, left, right, outer
- **Visualization**: Altair integrates directly with DataFrames

---

# A more realistic example

The [Sequence Read Archive](https://www.ncbi.nlm.nih.gov/sra) (SRA) is the largest public repository of sequencing data, mirrored by the European Nucleotide Archive (ENA). Here we analyze SARS-CoV-2 metadata to understand how sequencing platforms and library protocols were used during the pandemic.

## Setup

In [ ]:
import pandas as pd

We use a pre-compiled ENA metadata snapshot hosted on [Zenodo](https://zenodo.org/records/10680776). The file contains ~800k records; we load the first 100k for speed.

In [ ]:
sra = pd.read_csv(
    "https://zenodo.org/records/10680776/files/ena.tsv.gz",
    compression='gzip',
    sep="\t",
    low_memory=False,
    nrows=100000
)

## Explore the data

In [ ]:
len(sra)

In [ ]:
sra.sample(5)

In [ ]:
sra.columns.tolist()

In [ ]:
sra['instrument_platform'].value_counts()

## Clean dates

In [ ]:
# Convert collection_date to datetime
# errors='coerce' turns unparseable dates into NaT (Not a Time)
sra = sra.assign(collection_date=pd.to_datetime(sra['collection_date'], errors='coerce'))

In [ ]:
print('Earliest entry:', sra['collection_date'].min())
print('Latest entry:', sra['collection_date'].max())

> **⚠️ Data Quality:** Don't get surprised here — the metadata is only as good as the person who entered it. When you enter metadata for your sequencing data, pay attention!

In [ ]:
# Filter to valid date range
sra = sra[
    (sra['collection_date'] >= pd.Timestamp('2020-01-01'))
    &
    (sra['collection_date'] <= pd.Timestamp('2023-12-31'))
]

## Aggregate for visualization

In [ ]:
heatmap_2d = sra.groupby(
    ['instrument_platform', 'library_strategy']
).agg(
    {'run_accession': 'nunique'}
).reset_index()

heatmap_2d

## Visualize with Altair

In [ ]:
import altair as alt

In [ ]:
back = alt.Chart(heatmap_2d).mark_rect(opacity=1).encode(
    x=alt.X(
        "instrument_platform:N",
        title="Instrument"
    ),
    y=alt.Y(
        "library_strategy:N",
        title="Strategy",
        axis=alt.Axis(orient='right')
    ),
    color=alt.Color(
        "run_accession:Q",
        title="# Samples",
        scale=alt.Scale(
            scheme="goldred",
            type="log"
        ),
    ),
    tooltip=[
        alt.Tooltip("instrument_platform:N", title="Machine"),
        alt.Tooltip("run_accession:Q", title="Number of runs"),
        alt.Tooltip("library_strategy:N", title="Protocol")
    ]
).properties(
    width=500,
    height=150,
    title={
        "text": ["Breakdown of datasets from ENA",
                 "by Platform and Library Strategy"],
        "subtitle": "(Sample of 100k records)"
    }
)

back

In [ ]:
# Add text labels
front = back.mark_text(
    align="center",
    baseline="middle",
    fontSize=12,
    fontWeight="bold",
).encode(
    text=alt.Text("run_accession:Q", format=",.0f"),
    color=alt.condition(
        alt.datum.run_accession > 200,
        alt.value("white"),
        alt.value("black")
    )
)

# Combine layers
back + front